# 07 — Baseline e Regressão Logística

Executa `DummyClassifier` (piso) e `LogisticRegression` na partição
persistida pelo Notebook 02, usando **somente treino e validação**.

Sem tuning, sem calibração, sem escolha de limiar. O teste permanece
fechado. Limiar 0.5 é apenas diagnóstico.

**Memória:** o `DummyClassifier` roda com uma coluna trivial e **não usa o
preprocessor** (o `evaluate_baseline` do `src` roda o preprocessor mesmo
para o dummy, o que estoura a RAM em 556k × 28 categóricas). A logística
usa uma amostra reprodutível de 200 mil linhas para o ajuste; a validação
continua inteira. O tamanho amostral usado vai para o manifesto.


In [ ]:
from pathlib import Path
import gc
import json
import os
import sys

import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src import config as C
from src import config as env
from src.registro import Run
from src.dados import load_supervised_artifact, split_partitions
from src.avaliacao import check_leakage

env.carregar_dotenv()
env.garantir_diretorios()

COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "final").lower()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())

In [ ]:
import psutil

def mem_gb():
    return psutil.Process().memory_info().rss / 1e9

print(f"RAM antes do load: {mem_gb():.1f} GB")

df, manifest = load_supervised_artifact(ROOT_DIR, expected_kind=COHORT_KIND)
treino, validacao = split_partitions(df, manifest)

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

N_TREINO    = int(len(treino))
N_VALIDACAO = int(len(validacao))
N_TESTE     = int(sum(
    int(v) for k, v in manifest["partition_counts"].items()
    if str(k) in {"teste", "test_ref", "test_pandemic"}
))


def to_categoria(X: pd.DataFrame) -> pd.DataFrame:
    """Texto vira category: 556k × 28 strings passam a int8 + dicionário."""
    X = X.copy()
    for c in X.columns:
        if X[c].dtype == "object" or str(X[c].dtype).startswith("string"):
            X[c] = X[c].astype("category")
    return X


X_train = to_categoria(treino[FEATURES])
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8").copy()
X_val   = to_categoria(validacao[FEATURES])
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8").copy()

check_leakage(X_train)
check_leakage(X_val)

del df, treino, validacao
gc.collect()
try:
    import pyarrow as pa
    pa.default_memory_pool().release_unused()
except Exception:
    pass

print(f"RAM depois do load + categoria: {mem_gb():.1f} GB")
display(pd.DataFrame({
    "particao":   ["treino", "validacao"],
    "n":          [N_TREINO, N_VALIDACAO],
    "positivos":  [int(y_train.sum()), int(y_val.sum())],
    "prevalencia":[float(y_train.mean()), float(y_val.mean())],
}))
print(f"cohort_kind : {manifest['cohort_kind']}")
print(f"features    : {len(FEATURES)}")
print(f"teste       : {N_TESTE:,} (fechado)")

In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

from src.avaliacao import metricas_completas
from src.modelos import build_logistic_pipeline

X_trivial_train = np.zeros((len(y_train), 1), dtype="float32")
X_trivial_val   = np.zeros((len(y_val),   1), dtype="float32")

dummy = DummyClassifier(strategy="prior", random_state=42)
dummy.fit(X_trivial_train, y_train)
prob_val_dummy = np.asarray(dummy.predict_proba(X_trivial_val))[:, 1]
metricas_dummy = {
    **metricas_completas(y_val, prob_val_dummy, limiar=0.5),
    "modelo": "dummy_prior",
    "strategy": "prior",
}
del X_trivial_train, X_trivial_val
gc.collect()
print(f"[dummy] RAM {mem_gb():.2f} GB")

N_FIT_LOGISTIC = 200_000
if N_TREINO > N_FIT_LOGISTIC:
    X_fit, _, y_fit, _ = train_test_split(
        X_train, y_train,
        train_size=N_FIT_LOGISTIC,
        stratify=y_train,
        random_state=42,
    )
else:
    X_fit, y_fit = X_train, y_train

N_FIT_USADO = int(len(X_fit))
print(f"[logística] treino usado: {N_FIT_USADO:,} de {N_TREINO:,} "
      f"(prevalência {float(y_fit.mean()):.4f})")

log_cfg = C.cfg("models", "logistic") or {}
pipe_log = build_logistic_pipeline(
    C=1.0,
    solver=log_cfg.get("solver", "lbfgs"),
    max_iter=int(log_cfg.get("max_iter", 1000)),
    class_weight=log_cfg.get("class_weight", "balanced"),
)
pipe_log.fit(X_fit, y_fit)
prob_val_log = pipe_log.predict_proba(X_val)[:, 1]
metricas_log = {
    **metricas_completas(y_val, prob_val_log, limiar=0.5),
    "modelo": "logistica_C1",
}

del X_fit, y_fit
gc.collect()
print(f"[logística] RAM {mem_gb():.2f} GB")

display(pd.DataFrame([metricas_dummy, metricas_log]))

In [ ]:
CAMPOS = ("auc", "ap", "brier", "f1", "precision", "recall", "accuracy")

comparacao = (
    pd.DataFrame([
        {"modelo": "dummy_prior",   **{k: metricas_dummy[k] for k in CAMPOS}},
        {"modelo": "logistica_C1", **{k: metricas_log[k]   for k in CAMPOS}},
    ])
    .sort_values("ap", ascending=False)
    .reset_index(drop=True)
)

delta_ap = metricas_log["ap"] - metricas_dummy["ap"]
display(comparacao)
print(f"Δ AP (logística − dummy): {delta_ap:+.4f}")

pred_val = pd.DataFrame({
    "y_true":      y_val.to_numpy(),
    "y_prob":      prob_val_log,
    "modelo":      "logistica_C1",
    "cohort_kind": manifest["cohort_kind"],
})
pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo   = f"07_baseline_logistica_{manifest['cohort_kind']}"

caminho_comparacao = TABLES_DIR      / f"{prefixo}_comparacao.csv"
caminho_predicoes  = PREDICTIONS_DIR / f"{prefixo}_predicoes_validacao.parquet"
caminho_manifesto  = TABLES_DIR      / f"{prefixo}_manifesto.json"

comparacao.to_csv(caminho_comparacao, index=False, encoding="utf-8-sig")
pred_val.to_parquet(caminho_predicoes, index=False)

manifesto_c3 = {
    "cohort_kind":        manifest["cohort_kind"],
    "dataset_path":       manifest["dataset_path"],
    "dataset_sha256":     manifest["dataset_sha256"],
    "feature_columns":    FEATURES,
    "target_column":      ALVO,
    "partition_column":   PARTICAO,
    "n_treino":           N_TREINO,
    "n_validacao":        N_VALIDACAO,
    "n_teste_fechado":    N_TESTE,
    "n_fit_logistica":    N_FIT_USADO,
    "limiar_usado":       0.5,
    "limiar_e_diagnostico": True,
    "tuning_realizado":   False,
    "calibracao_aplicada":False,
    "teste_aberto":       False,
    "modelos":            ["dummy_prior", "logistica_C1"],
    "artefatos": {
        "comparacao":          str(caminho_comparacao),
        "predicoes_validacao": str(caminho_predicoes),
    },
}
caminho_manifesto.write_text(
    json.dumps(manifesto_c3, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Comparação : {caminho_comparacao}")
print(f"Predições  : {caminho_predicoes}")
print(f"Manifesto  : {caminho_manifesto}")

In [ ]:
with Run(
    modelo="07_baseline_logistica",
    parametros={
        "cohort_kind":          manifest["cohort_kind"],
        "logistic_C":           1.0,
        "logistic_solver":      log_cfg.get("solver", "lbfgs"),
        "limiar_diagnostico":   0.5,
        "tuning_realizado":     False,
        "calibracao_aplicada":  False,
        "teste_aberto":         False,
        "n_fit_logistica":      N_FIT_USADO,
    },
    dataset={
        "n_treino":        N_TREINO,
        "n_validacao":     N_VALIDACAO,
        "n_teste_fechado": N_TESTE,
    },
    features={"n_features": len(FEATURES), "usadas": FEATURES},
    notas=(
        "C3 — baseline e logística na partição persistida. "
        "Dummy sem preprocessor; logística ajustada em amostra de 200k linhas. "
        "Teste não consultado."
    ),
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(
        dummy_ap=metricas_dummy["ap"],
        dummy_auc=metricas_dummy["auc"],
        logistica_ap=metricas_log["ap"],
        logistica_auc=metricas_log["auc"],
        logistica_brier=metricas_log["brier"],
    )

print("Run registrado em:", r.pasta)